In [1]:


import numpy as np
from numpy.linalg import matrix_power
#import sympy as sym
from scipy import optimize
# from scipy.optimize import fsolve
import ipywidgets as widgets
import networkx as nx
import matplotlib.gridspec as gridspec


import jax
import jax.numpy as jnp
from scipy import optimize, linalg

import mpl_toolkits
from matplotlib import projections
from matplotlib import pyplot as plt
from scipy.integrate import odeint
from scipy.integrate import solve_ivp
from scipy.optimize import linear_sum_assignment


# =============================================================================
def ODEs_rE0_optimize(var, kwargs):
    x0   = var

    wEE       = kwargs['wEE']
    wEI       = kwargs['wEI']
    wIE       = kwargs['wIE']
    wII       = kwargs['wII']
    
    wEe       = kwargs['wEe']
    wEi       = kwargs['wEi']
    wIe       = kwargs['wIe']
    wIi       = kwargs['wIi']
    
    
   
    # rI0       = kwargs['rI0']
    Lb        = kwargs['Lb']
    alpha     = kwargs['alpha']
    tauE      = kwargs['tauE']
    
    rE0 = x0
    rI0 = kwargs['rI0']
    
    iE0     = (alpha)*(Lb) 
    iI0     = (1-alpha)*(Lb)
    
    WEE, WEI, WII, WIE = wEE + 2*wEe,  wEI + 2*wEi, wII + 2*wIi, wIE + 2*wIe
    DEE, DEI, DII, DIE = wEe, wEi, wIi, wIe
    X0 = rE0*WEE-rI0*WEI+iE0
    Y0 = rE0*WIE-rI0*WII+iI0

    # X0      = rE0*wEE-rI0*wEI+iE
    rE0out  = (1/tauE)*(-rE0+np.tanh(X0))
    
    return rE0out
"""
||$||
"""
def ODEs_rI0_optimize(var, kwargs):
    # t    = var[0]
    # x0   = var[1]
    x1   = var

    wEE       = kwargs['wEE']
    wEI       = kwargs['wEI']
    wIE       = kwargs['wIE']
    wII       = kwargs['wII']
    
    wEe       = kwargs['wEe']
    wEi       = kwargs['wEi']
    wIe       = kwargs['wIe']
    wIi       = kwargs['wIi']
    
    # rE0       = kwargs['rE0']
    Lb        = kwargs['Lb']
    alpha     = kwargs['alpha']
    tauE      = kwargs['tauE']

    rE0 = kwargs['rE0']
    rI0 = x1

    iE0= alpha*Lb
    iI0= (1-alpha)*Lb
    
    WEE, WEI, WII, WIE = wEE + 2*wEe,  wEI + 2*wEi, wII + 2*wIi, wIE + 2*wIe
    DEE, DEI, DII, DIE = wEe, wEi, wIi, wIe
    X0 = rE0*WEE-rI0*WEI+iE0
    Y0 = rE0*WIE-rI0*WII+iI0
    
    # Y0      = rE0*wIE-rI0*wII+iI0
    rI0out  = -rI0+np.tanh(Y0)
    return rI0out

def coupledODEs_rE0_rI0(var, kwargs):
    x0   = var[0]
    x1   = var[1]

    wEE       = kwargs['wEE']
    wEI       = kwargs['wEI']
    wIE       = kwargs['wIE']
    wII       = kwargs['wII']
    
    wEe       = kwargs['wEe']
    wEi       = kwargs['wEi']
    wIe       = kwargs['wIe']
    wIi       = kwargs['wIi']
    
    Lb        = kwargs['Lb']
    alpha     = kwargs['alpha']
    tauE      = kwargs['tauE']

    rE0 = x0
    rI0 = x1
    
    iE0 = alpha*Lb
    iI0 = (1-alpha)*Lb
    
    WEE, WEI, WII, WIE = wEE + 2*wEe,  wEI + 2*wEi, wII + 2*wIi, wIE + 2*wIe
    DEE, DEI, DII, DIE = wEe, wEi, wIi, wIe
    
    X0 = rE0*WEE-rI0*WEI+iE0
    Y0 = rE0*WIE-rI0*WII+iI0

    # tauS = 0.001

    rE0out  = (-rE0+np.tanh(X0))/(tauE*tauS)
    rI0out  = (-rI0+np.tanh(Y0))/(tauS)
    return [rE0out, rI0out]

def preferredk_r0_abc_safe(kwargs, eps=1e-6, debug=False):

    # ---- unpack ----
    wEE = kwargs['wEE']; wEI = kwargs['wEI']
    wIE = kwargs['wIE']; wII = kwargs['wII']

    wEe = kwargs['wEe']; wEi = kwargs['wEi']
    wIe = kwargs['wIe']; wIi = kwargs['wIi']

    rE0 = kwargs['rE0']
    rI0 = kwargs['rI0']

    # ---- derived weights ----
    WEE = wEE + 2*wEe
    WEI = wEI + 2*wEi
    WII = wII + 2*wIi
    WIE = wIE + 2*wIe

    DEE, DEI, DII, DIE = wEe, wEi, wIi, wIe

    # =========================
    # SAFE REGULARISATION
    # =========================
    denomE = 1 - rE0**2
    denomI = 1 - rI0**2

    # smooth "closest valid value"
    denomE_safe = denomE + eps * np.sign(denomE if denomE != 0 else 1.0)
    denomI_safe = denomI + eps * np.sign(denomI if denomI != 0 else 1.0)

    RE0 = 1.0 / denomE_safe
    RI0 = 1.0 / denomI_safe

    # optional: cap extreme blowups (prevents downstream instability)
    MAX_VAL = 1e6
    RE0 = np.clip(RE0, -MAX_VAL, MAX_VAL)
    RI0 = np.clip(RI0, -MAX_VAL, MAX_VAL)

    # =========================
    # COMPUTE COEFFICIENTS
    # =========================
    A0 = RE0 - WEE
    D0 = RI0 + WII

    a = (-DEE * DII) + (DIE * DEI)
    b = (DEE * D0) - (DII * A0) - (DIE * WEI) - (DEI * WIE)
    c = (A0 * D0) + (WIE * WEI)

    # =========================
    # FINAL SAFETY (guarantee return)
    # =========================
    if not np.isfinite(a): a = 0.0
    if not np.isfinite(b): b = 0.0
    if not np.isfinite(c): c = 0.0

    out4 = DEE * D0 if np.isfinite(DEE * D0) else 0.0
    out5 = (-DII * A0) if np.isfinite(-DII * A0) else 0.0

    if debug:
        print(f"rE0={rE0:.3f}, rI0={rI0:.3f}, RE0={RE0:.3f}, RI0={RI0:.3f}")

    return a, b, c, out4, out5

def coupledODEs_rE0_rI0(var, kwargs):
    x0 = var[0]; x1 = var[1]
    wEE = kwargs['wEE']; wEI = kwargs['wEI']; wIE = kwargs['wIE']; wII = kwargs['wII']
    wEe = kwargs['wEe']; wEi = kwargs['wEi']; wIe = kwargs['wIe']; wIi = kwargs['wIi']
    Lb = kwargs['Lb']; alpha = kwargs['alpha']; tauE = kwargs.get('tauE', 1.0)

    rE0 = x0; rI0 = x1
    iE0 = alpha * Lb
    iI0 = (1.0 - alpha) * Lb

    WEE, WEI, WII, WIE = wEE + 2 * wEe, wEI + 2 * wEi, wII + 2 * wIi, wIE + 2 * wIe
    DEE, DEI, DII, DIE = wEe, wEi, wIi, wIe

    X0 = rE0 * WEE - rI0 * WEI + iE0
    Y0 = rE0 * WIE - rI0 * WII + iI0

    rE0out = (-rE0 + np.tanh(X0)) / (tauE)
    rI0out = (-rI0 + np.tanh(Y0))
    return [rE0out, rI0out]


def jacobian_numeric(func, x, args=(), eps=1e-6):
    x = np.asarray(x, dtype=float)
    n = x.size
    J = np.zeros((n, n), dtype=float)
    f0 = np.asarray(func(x, *args), dtype=float)
    for i in range(n):
        dx = np.zeros(n)
        dx[i] = eps
        f1 = np.asarray(func(x + dx, *args), dtype=float)
        f2 = np.asarray(func(x - dx, *args), dtype=float)
        J[:, i] = (f1 - f2) / (2.0 * eps)
    return J


def rErI_eq0_simulation(kwargs):
    """
    Find a stable equilibrium for given kwargs.
    Uses previous-state guess provided in kwargs['rE0'], kwargs['rI0'] if possible.
    Returns (rE, rI) floats.
    """
    rE_prev = float(kwargs.get('rE0', 0.0))
    rI_prev = float(kwargs.get('rI0', 0.0))

    cand = []
    # try the "smart" single guess first
    try:
        sol = optimize.root(coupledODEs_rE0_rI0, [rE_prev, rI_prev], args=(kwargs,), tol=1e-10)
        if sol.success:
            cand.append(sol.x)
    except Exception:
        pass

    # fallback: grid of guesses
    if not cand:
        guesses = np.linspace(-1.5, 1.5, 9)  # tighter grid - faster and usually sufficient
        for rE_guess in guesses:
            for rI_guess in guesses:
                try:
                    sol = optimize.root(coupledODEs_rE0_rI0, [rE_guess, rI_guess], args=(kwargs,), tol=1e-10)
                    if sol.success:
                        cand.append(sol.x)
                except Exception:
                    continue

    if not cand:
        # ultimate fallback -> hold last state
        return rE_prev, rI_prev

    # deduplicate
    uniq = []
    tol = 1e-4
    for r in cand:
        if not any(npl.norm(r - u) < tol for u in uniq):
            uniq.append(r)
    uniq = np.array(uniq)

    # stability check via numeric Jacobian
    stable = []
    for r in uniq:
        try:
            J = jacobian_numeric(coupledODEs_rE0_rI0, r, args=(kwargs,))
            ev = npl.eigvals(J)
            if np.all(np.real(ev) < 0):
                stable.append(r)
        except Exception:
            # if Jacobian failed, keep as candidate but not stable
            pass

    chosen = None
    if stable:
        # choose stable root closest to previous state (keeps continuity)
        dists = [npl.norm(s - np.array([rE_prev, rI_prev])) for s in stable]
        chosen = stable[int(np.argmin(dists))]
    else:
        # fallback to the uniq root closest to previous
        dists = [npl.norm(u - np.array([rE_prev, rI_prev])) for u in uniq]
        chosen = uniq[int(np.argmin(dists))]

    return float(chosen[0]), float(chosen[1])

def simulate_equilibria(params, xFit):
    """
    Simulate equilibria across xFit sequentially, passing previous equilibrium as initial guess.
    This is intentionally simple and avoids caches to be stable under multiprocessing.
    params: array-like [wEE, wEe, wEI, wEi, wIE, wIe, wII, wIi]
    """
    rE_last, rI_last = 0.0, 0.0
    rE_hist, rI_hist = [], []
    for LB in xFit:
        rE_last, rI_last = rErI_eq0_simulation({
            'wEE': params[0], 'wEI': params[2], 'wIE': params[4], 'wII': params[6],
            'wEe': params[1], 'wEi': params[3], 'wIe': params[5], 'wIi': params[7],
            'Lb': float(LB), 'alpha': params[8], 'tauE': 1.0,
            'rE0': rE_last, 'rI0': rI_last
        })
        rE_hist.append(rE_last); rI_hist.append(rI_last)
    return np.array(rE_hist), np.array(rI_hist)

def compute_quasi_weights(weights , rE0, rI0):
    """
    omega        : dict {('E','E'):..., ('E','I'):..., ...}  local weights
    tilde_omega  : dict same shape, neighbour weights
    rE0, rI0     : background firing rates under luminance L
    """

    R_E0 = 1.0 / (1 - rE0**2)
    R_I0 = 1.0 / (1 - rI0**2)
    
    wEE, wEe, wEI, wEi, wIE, wIe, wII, wIi =  weights
    # Effective on-site weights (big loops)
    wEEn = R_E0*(wEE + 2*wEe)
    wEIn = R_E0*(wEI + 2*wEi)
    wIEn = R_I0*(wIE + 2*wIe)
    wIIn = R_I0*(wII + 2*wIi)

    # Effective neighbour-to-central couplings (thin edges)
    wEen = R_E0*wEe
    wEin = R_E0*wEi
    wIen = R_I0*wIe
    wIin = R_I0*wIi

    return [wEEn,wEen,wEIn,wEin,wIEn,wIen,wIIn,wIin]

def findCorrespondingValue_Similar(xList, yList, findValue, bounds):
    xfit_max = 0
    boundList= list(np.arange(findValue-bounds, findValue+bounds, 0.1))
    for i in range(len(yList)):
        if yList[i] in boundList:
            xfit_max = xList[i]
        elif yList[i] not in boundList:
            pass
    return xfit_max

def jacobian_jax(func, x, args=()):
    return jax.jacfwd(func)(x, *args)

""""
def numerical_jacobian(func, x, args=(), eps=1e-6):
    
    #Approximate Jacobian of vector function func at point x.
    #func(x, *args) should return a 1D array of length n.
    
    x  = np.array(x, dtype=float)
    f0 = np.array(func(x, *args))
    n  = len(x)
    J  = np.zeros((n, n))
    for i in range(n):
        dx    = np.zeros(n)
        dx[i] = eps
        f1    = np.array(func(x + dx, *args))
        J[:, i] = (f1 - f0) / eps
    return J
"""

def numerical_jacobian(func, x, args=(), eps=None):
    """
    Robust central-difference numerical Jacobian.
    func(x, *args) should return array-like (m,) for x in R^n.
    
    Returns: Jacobian of shape (m, n).
    """
    x  = np.array(x, dtype=float)
    f0 = np.array(func(x, *args), dtype=float)
    n  = len(x)
    m  = len(f0)
    J  = np.zeros((m, n))

    for i in range(n):
        # adaptive step size
        h = eps if eps is not None else np.sqrt(np.finfo(float).eps) * (1 + abs(x[i]))
        dx = np.zeros(n)
        dx[i] = h
        f_plus  = np.array(func(x + dx, *args), dtype=float)
        f_minus = np.array(func(x - dx, *args), dtype=float)
        J[:, i] = (f_plus - f_minus) / (2*h)

    return J

def find_all_roots(f, x_min=-2.0, x_max=2.0, npts=2001):
    """
    Find all roots of f in the interval [x_min, x_max] by sampling on a grid
    and using a root-finding method (Brent's method) in intervals where a sign
    change is detected.
    """
    xs = np.linspace(x_min, x_max, npts)
    fs = np.empty_like(xs)
    fs[:] = np.nan
    for i, x in enumerate(xs):
        try:
            fs[i] = float(f(x))
        except Exception:
            pass

    roots = []
    for i in range(len(xs) - 1):
        if np.isnan(fs[i]) or np.isnan(fs[i+1]):
            continue
        if fs[i] == 0.0:
            roots.append(xs[i])
        elif fs[i] * fs[i+1] < 0:  # sign change → root exists
            try:
                r = optimize.brentq(f, xs[i], xs[i+1], xtol=1e-10, rtol=1e-10)
                roots.append(r)
            except Exception:
                pass
    return roots

def find_nearest(array, value):
    array = np.asarray(array)

    if array.size == 0:
        return None  # <-- critical fix

    idx = (np.abs(array - value)).argmin()
    return array[idx]
# =============================================================================
def plotKwargs(**kwargs):

    weights    = [kwargs['wEE'], kwargs['wEe'], kwargs['wEI'], kwargs['wEi'], 
                  kwargs['wIE'], kwargs['wIe'], kwargs['wII'], kwargs['wIi']]

    jDC        = kwargs['jDCV']
    maxLb      = kwargs['maxLbv']
    alpha      = kwargs['alpha']
    tauE       = kwargs['tauE']
    dstepLb    = kwargs['stepLb']

    wEE, wEe, wEI, wEi, wIE, wIe, wII, wIi = weights 

    listLuminace = np.arange(0, maxLb, dstepLb)

    l           = 0
    chosenk     = 2.0
    Amplitude   = 0.1

    rE_all, rI_all = [], []
    rE_all2, rI_all2 = [], []
    
    stable_points_np   = np.empty((0,2))
    unstable_points_np = np.empty((0,2))

    # =========================
    # STORAGE (NEW CLEAN SYSTEM)
    # =========================
    branches = []
    prev_roots = None

    # =========================
    # MAIN LOOP
    # =========================
    for Lb in listLuminace:

        kwargsODE = {
            'wEE':wEE, 'wEI':wEI, 'wIE':wIE, 'wII':wII,
            'wEe':wEe, 'wEi':wEi,'wIe':wIe, 'wIi':wIi,
            'Amplitude':Amplitude,
            'Lb':Lb,'alpha':alpha,'tauE':tauE,
            'rE0':0.0,'rI0':0.0,
            'l':l,'k':chosenk,
        }

        # =========================
        # FIND ROOTS
        # =========================
        roots = []
        for rE_guess in np.linspace(-2, 2, 10):
            for rI_guess in np.linspace(-2, 2, 10):
                try:
                    sol = optimize.root(coupledODEs_rE0_rI0,
                                        [rE_guess, rI_guess],
                                        args=(kwargsODE,),
                                        tol=1e-10)
                    if sol.success:
                        roots.append(sol.x)
                except:
                    pass

        # deduplicate
        unique_roots = []
        tol = 1e-3
        for r in roots:
            if not any(np.linalg.norm(r - u) < tol for u in unique_roots):
                unique_roots.append(r)

        unique_roots = np.array(unique_roots)

        # =========================
        # STABILITY
        # =========================
        all_roots = []
        all_stability = []

        for r in unique_roots:
            J = numerical_jacobian(coupledODEs_rE0_rI0, r, args=(kwargsODE,))
            eigvals = linalg.eigvals(J)

            if np.all(np.real(eigvals) < 0):
                all_roots.append(r)
                all_stability.append(1)
            else:
                all_roots.append(r)
                all_stability.append(0)

        all_roots = np.array(all_roots)

        if len(all_roots) == 0:
            continue

        # =========================
        # FIRST STEP
        # =========================
        if prev_roots is None:

            prev_roots = all_roots.copy()

            for i in range(len(all_roots)):
                rE, rI = all_roots[i]
                stab   = all_stability[i]

                branches.append({
                    'Lb': [Lb],
                    'rE': [rE],
                    'rI': [rI],
                    'stable': [stab],
                    'a': [],
                    'b': [],
                    'c': []
                })
                a,b,c,_,_ = preferredk_r0_abc_safe({
                    'wEE':wEE, 'wEI':wEI, 'wIE':wIE, 'wII':wII,
                    'wEe':wEe, 'wEi':wEi,'wIe':wIe, 'wIi':wIi,
                    'Amplitude':Amplitude,
                    'Lb':Lb,'alpha':alpha,'tauE':tauE,
                    'rE0':rE,'rI0':rI,
                    'l':l,'k':chosenk,
                })
                
                branches[i]['a'].append(a)
                branches[i]['b'].append(b)
                branches[i]['c'].append(c)

        # =========================
        # MATCHING (HUNGARIAN)
        # =========================
        else:

            tol_match = 0.5

            n_prev = len(prev_roots)
            n_new  = len(all_roots)

            cost = np.zeros((n_prev, n_new))

            for i in range(n_prev):
                for j in range(n_new):
                    cost[i, j] = np.linalg.norm(prev_roots[i] - all_roots[j])

            row_ind, col_ind = linear_sum_assignment(cost)

            used_prev = set()
            used_new  = set()

            new_prev_roots = [None]*n_prev

            # =========================
            # MATCHED
            # =========================
            for i, j in zip(row_ind, col_ind):

                if cost[i, j] > tol_match:
                    continue

                used_prev.add(i)
                used_new.add(j)

                rE, rI = all_roots[j]
                stab   = all_stability[j]

                branches[i]['Lb'].append(Lb)
                branches[i]['rE'].append(rE)
                branches[i]['rI'].append(rI)
                branches[i]['stable'].append(stab)

                kwargsK = {
                    'wEE':wEE, 'wEI':wEI, 'wIE':wIE, 'wII':wII,
                    'wEe':wEe, 'wEi':wEi,'wIe':wIe, 'wIi':wIi,
                    'Amplitude':Amplitude,
                    'Lb':Lb,'alpha':alpha,'tauE':tauE,
                    'rE0':rE,'rI0':rI,
                    'l':l,'k':chosenk,
                }
            
                a,b,c,_,_ = preferredk_r0_abc_safe(kwargsK)
            
                branches[i]['a'].append(a)
                branches[i]['b'].append(b)
                branches[i]['c'].append(c)
            
                new_prev_roots[i] = [rE, rI]

            # =========================
            # UNMATCHED OLD → NaN
            # =========================
            for i in range(n_prev):
                if i not in used_prev:

                    branches[i]['Lb'].append(Lb)
                    branches[i]['rE'].append(np.nan)
                    branches[i]['rI'].append(np.nan)
                    branches[i]['stable'].append(np.nan)
                    
                    branches[i]['a'].append(np.nan)
                    branches[i]['b'].append(np.nan)
                    branches[i]['c'].append(np.nan)

                    new_prev_roots[i] = prev_roots[i]

            # =========================
            # NEW ROOTS → NEW BRANCHES
            # =========================
            for j in range(n_new):
                if j not in used_new:

                    rE, rI = all_roots[j]
                    stab   = all_stability[j]

                    kwargsK = {
                        'wEE':wEE, 'wEI':wEI, 'wIE':wIE, 'wII':wII,
                        'wEe':wEe, 'wEi':wEi,'wIe':wIe, 'wIi':wIi,
                        'Amplitude':Amplitude,
                        'Lb':Lb,'alpha':alpha,'tauE':tauE,
                        'rE0':rE,'rI0':rI,
                        'l':l,'k':chosenk,
                    }
                    
                    a,b,c,_,_ = preferredk_r0_abc_safe(kwargsK)
                    
                    branches.append({
                        'Lb': [Lb],
                        'rE': [rE],
                        'rI': [rI],
                        'stable': [stab],
                        'a': [a],
                        'b': [b],
                        'c': [c]
                    })

                    new_prev_roots.append([rE, rI])

            prev_roots = np.array(new_prev_roots)

    # =========================
    # FILTER VALID BRANCHES
    # =========================
    valid_branches = []
    
    for b in branches:
        rE = np.array(b['rE'])
        if len(rE) == 0 or np.all(np.isnan(rE)):
            continue
        valid_branches.append(b)
    
    n = len(valid_branches)
    
    if n == 0:
        print("No valid branches found.")
        return

    colors = plt.cm.viridis(np.linspace(0, 1, len(valid_branches)))
    # =========================
    # NULLCLINES AT jDC
    # =========================
    kwargs_null = {
        'wEE':wEE, 'wEI':wEI, 'wIE':wIE, 'wII':wII,
        'wEe':wEe, 'wEi':wEi,'wIe':wIe, 'wIi':wIi,
        'Amplitude':Amplitude,
        'Lb':jDC,'alpha':alpha,'tauE':tauE,
        'rE0':0.0,'rI0':0.0,
        'l':l,'k':chosenk,
    }
        
    # =========================
    # NULLCLINES + FIXED POINTS @ jDC
    # =========================
    
    stepSize = 0.05
    
    # --- rE nullcline ---
    rI_grid = np.arange(-2, 2, stepSize)
    rE_all, rI_all = [], []
    
    for rI_val in rI_grid:
        kwargs_local = dict(kwargs_null)
        kwargs_local['rI0'] = rI_val
    
        f = lambda rE: ODEs_rE0_optimize(rE, kwargs_local)
    
        roots = find_all_roots(f, -2, 2)
        for r in roots:
            rE_all.append(r)
            rI_all.append(rI_val)
    
    # --- rI nullcline ---
    rE_grid = np.arange(-2, 2, stepSize)
    rI_all2, rE_all2 = [], []
    
    for rE_val in rE_grid:
        kwargs_local = dict(kwargs_null)
        kwargs_local['rE0'] = rE_val
    
        f = lambda rI: ODEs_rI0_optimize(rI, kwargs_local)
    
        roots = find_all_roots(f, -2, 2)
        for r in roots:
            rI_all2.append(r)
            rE_all2.append(rE_val)
    
    # =========================
    # FIXED POINTS (ADD THIS)
    # =========================
    roots = []
    
    for rE_guess in np.linspace(-2, 2, 10):
        for rI_guess in np.linspace(-2, 2, 10):
            try:
                sol = optimize.root(coupledODEs_rE0_rI0,
                                    [rE_guess, rI_guess],
                                    args=(kwargs_null,),
                                    tol=1e-12)
                if sol.success:
                    roots.append(sol.x)
            except:
                pass
    
    roots = np.array(roots)
    
    # --- deduplicate ---
    unique_roots = []
    tol = 1e-3
    
    for r in roots:
        if not any(np.linalg.norm(r - u) < tol for u in unique_roots):
            unique_roots.append(r)
    
    unique_roots = np.array(unique_roots)
    
    # --- stability ---
    stable_points = []
    unstable_points = []
    
    for r in unique_roots:
        try:
            J = numerical_jacobian(coupledODEs_rE0_rI0, r, args=(kwargs_null,))
            eigvals = linalg.eigvals(J)
    
            if np.all(np.real(eigvals) < 0):
                stable_points.append(r)
            else:
                unstable_points.append(r)
        except:
            pass
    
    # convert safely
    stable_points_np   = np.array(stable_points)   if len(stable_points) > 0 else np.empty((0,2))
    unstable_points_np = np.array(unstable_points) if len(unstable_points) > 0 else np.empty((0,2))
    
    # =========================
    # GRID LAYOUT
    # =========================
    n_cols_left = int(np.ceil(np.sqrt(n)))
    n_rows_left = int(np.ceil(n / n_cols_left))
    
    fig = plt.figure(figsize=(18, 12))
    
    gs_outer = gridspec.GridSpec(
        1, 3,
        width_ratios=[2, 2.5, 1.5],  # left / middle / right
        wspace=0.3
    )
    gs_left = gridspec.GridSpecFromSubplotSpec(
        3, 2,   # ← FORCE 2 cols × 3 rows
        subplot_spec=gs_outer[0],
        wspace=0.3,
        hspace=0.3
    )

    # =========================
    # LEFT: BRANCH PLOTS
    # =========================
    for i, b in enumerate(valid_branches):
        color = colors[i] 
        row = i // 2
        col = i % 2
        ax = fig.add_subplot(gs_left[row, col])
    
        Lb = np.array(b['Lb'])
        rE = np.array(b['rE'])
        rI = np.array(b['rI'])
        st = np.array(b['stable'])
    
        mask = np.isfinite(rE)
    
        Lb = Lb[mask]
        rE = rE[mask]
        rI = rI[mask]
        st = st[mask]
    
        if len(Lb) == 0:
            continue
    
        mask_s = st == 1
        mask_u = st == 0
    
        ax.plot(Lb[mask_s], rE[mask_s], 'r.-', label='rE')
        ax.plot(Lb[mask_s], rI[mask_s], 'b.-', label='rI')
    
        ax.plot(Lb[mask_u], rE[mask_u], 'r--')
        ax.plot(Lb[mask_u], rI[mask_u], 'b--')

        #ax.plot(Lb[mask_s], rE[mask_s], color=color, linestyle='-', marker='.')
        #ax.plot(Lb[mask_s], rI[mask_s], color=color, linestyle='-', marker='.')
    
        #ax.plot(Lb[mask_u], rE[mask_u], color=color, linestyle='--')
        #ax.plot(Lb[mask_u], rI[mask_u], color=color, linestyle='--')
    
        # COLOUR THE BORDER
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3.5)
    
        ax.axvline(jDC, color='green')
        ax.axhline(0, color='black')
    
        ax.set_title(f'Branch {i}', fontsize=10)
        ax.set_xlim(0, maxLb)
        ax.set_ylim(-1.2, 1.2)
        ax.set_box_aspect(1)
    
    # =========================
    # RIGHT TOP: NULLCLINES
    # =========================
    ax_null = fig.add_subplot(gs_outer[1])    
    if len(rE_all) > 0:
        ax_null.scatter(rE_all, rI_all, s=2, c='red', label='rI nullcline')
    
    if len(rE_all2) > 0:
        ax_null.scatter(rE_all2, rI_all2, s=2, c='blue', label='rE nullcline')
    
    # =========================
    # PLOT FIXED POINTS (ROBUST)
    # =========================
    if stable_points_np.shape[0] > 0:
        ax_null.scatter(stable_points_np[:,0], stable_points_np[:,1],
                        s=80, c='green', marker='o', label='Stable')
    
    if unstable_points_np.shape[0] > 0:
        ax_null.scatter(unstable_points_np[:,0], unstable_points_np[:,1],
                        s=80, c='orange', marker='x', label='Unstable')
    
    ax_null.set_xlim(-2, 2)
    ax_null.set_ylim(-2, 2)
    ax_null.set_title('Nullclines')
    ax_null.set_box_aspect(1)

    ax_null.set_xlabel(r'$r_E$')
    ax_null.set_ylabel(r'$r_I$')
    ax_null.set_title('Nullclines')
    ax_null.set_box_aspect(1)
    
    # =========================
    # RIGHT BOTTOM: ISF (COLOURED BY BRANCH)
    # =========================
    
    #ax_isf = fig.add_subplot(gs[n_rows_left//2:, -1])
    
    # get a nice colour map

    gs_right = gridspec.GridSpecFromSubplotSpec(
        2, 1,
        subplot_spec=gs_outer[2],
        hspace=0.3
    )
    
    ax_kn1 = fig.add_subplot(gs_right[0])
    ax_kn2 = fig.add_subplot(gs_right[1])
    for i, b in enumerate(branches):

        color = colors[i]
    
        Lb = np.array(b['Lb'], dtype=float)
        a  = np.array(b['a'], dtype=float)
        b_ = np.array(b['b'], dtype=float)
        c  = np.array(b['c'], dtype=float)
        st = np.array(b['stable'], dtype=float)
    
        min_len = min(len(Lb), len(a), len(b_), len(c), len(st))
        if min_len == 0:
            continue
    
        Lb = Lb[:min_len]
        a  = a[:min_len]
        b_ = b_[:min_len]
        c  = c[:min_len]
        st = st[:min_len]
    
        disc = np.lib.scimath.sqrt(b_**2 - 4*a*c)
    
        denom = 2*a
        denom[np.abs(denom) < 1e-8] = 1e-8

        with np.errstate(invalid='ignore', divide='ignore'):
            kn1 = np.real(np.lib.scimath.sqrt((-b_ - disc) / denom))
            kn2 = np.real(np.lib.scimath.sqrt((-b_ + disc) / denom))
    
        mask1 = np.isfinite(kn1) & (kn1 > 0) & (kn1 < 20)
        mask2 = np.isfinite(kn2) & (kn2 > 0) & (kn2 < 20)
    
        # -------- kn1 plot --------
        ax_kn1.scatter(Lb[mask1 & (st==1)], kn1[mask1 & (st==1)],
                       c=[color], s=12)
    
        ax_kn1.scatter(Lb[mask1 & (st==0)], kn1[mask1 & (st==0)],
                       c=[color], marker='x', s=12)
    
        ax_kn1.plot(Lb[mask1], kn1[mask1], color=color, alpha=0.3)
    
        # -------- kn2 plot --------
        ax_kn2.scatter(Lb[mask2 & (st==1)], kn2[mask2 & (st==1)],
                       c=[color], s=12)
    
        ax_kn2.scatter(Lb[mask2 & (st==0)], kn2[mask2 & (st==0)],
                       c=[color], marker='x', s=12)
    
        ax_kn2.plot(Lb[mask2], kn2[mask2], color=color, alpha=0.3)
        # -------------------------
        # FORMAT
        # -------------------------
    for ax, title in zip([ax_kn1, ax_kn2], ['k_n1', 'k_n2']):
        ax.axvline(jDC, color='green')
        ax.set_xlim(0, maxLb)
        ax.set_ylim(0, 16)
        ax.set_xlabel('Lb')
        ax.set_ylabel('k*')
        ax.set_title(title)
        ax.set_box_aspect(1)
    plt.show()

### === ###

style = {'description_width': 'initial'}
layout= widgets.Layout(width='200px')    

wEEv      = widgets.FloatSlider(value=8.0, description='wEE', max=20.0, step=0.05, min=-10.0,layout=layout)
wEev      = widgets.FloatSlider(value=6.4, description='wEe', max=20.0, step=0.05, min=-10.0,layout=layout)
wEIv      = widgets.FloatSlider(value=4.0, description='wEI', max=20.0, step=0.05, min=-10.0,layout=layout)
wEiv      = widgets.FloatSlider(value=4.00, description='wEi', max=20.0, step=0.05, min=-10.0,layout=layout)
wIEv      = widgets.FloatSlider(value=7.20, description='wIE', max=20.0, step=0.05, min=-10.0,layout=layout)
wIev      = widgets.FloatSlider(value=1.00, description='wIe', max=20.0, step=0.05, min=-10.0,layout=layout)
wIIv      = widgets.FloatSlider(value=1.0, description='wII', max=20.0, step=0.05, min=-10.0,layout=layout)
wIiv      = widgets.FloatSlider(value=0.1, description='wIi', max=20.0, step=0.05, min=-10.0,layout=layout) 

layout1= widgets.Layout(width='500px')    
jDCV      = widgets.FloatSlider(value=18.0, description='Lb-check', max=100.0, step=0.1, min=0.0,layout=layout1)

alphaV    = widgets.FloatSlider(value=0.5, description='alpha', max=1.0, step=0.05, min=0.0,layout=layout)
tauEV     = widgets.FloatSlider(value=1.0, description='tauE', max=5.0, step=0.01, min=0.01,layout=layout)
stepLbv   = widgets.FloatSlider(value=0.1, description='Lb-step', max=1, step=0.01, min=0.001,layout=layout)
maxLbv    = widgets.IntSlider(value=60.0, description='Lb-max', max=100, step=10, min=10,layout=layout)



variables = {'alpha':alphaV,'tauE':tauEV,'stepLb':stepLbv,
                'wEE':wEEv,'wEe':wEev,'wEI':wEIv,'wEi':wEiv,
                'wIE':wIEv,'wIe':wIev,'wII':wIIv,'wIi':wIiv,
             'jDCV':jDCV,'maxLbv':maxLbv
            }

ui0       = widgets.HBox([alphaV,tauEV,stepLbv, maxLbv, jDCV])

ui1       = widgets.VBox([wEev, wEiv,wIev, wIiv])
ui2       = widgets.VBox([wEEv, wEIv,wIEv, wIIv])

wghtV     = widgets.VBox([ui1, ui2])

out       = widgets.interactive_output(plotKwargs, variables)



outH       = widgets.HBox([out, wghtV])
display(widgets.VBox([ui0,outH])) 

  